In [0]:
# Incremental Ingestion Demo (Bronze Layer) using Auto Loader

#   - Day1 ingestion
#   - Day2 ingestion (new + updated + duplicate records)
#   - Day3 ingestion (schema evolution)

In [0]:
# Base incremental folder
incremental_path = "/Volumes/ecommerce/bronze/raw_files/incremental"

# Checkpoint folder for Auto Loader
checkpoint_path = "/Volumes/ecommerce/bronze/raw_files/checkpoints/incremental_orders"

print("Incremental Path:", incremental_path)
print("Checkpoint Path:", checkpoint_path)


Incremental Path: /Volumes/ecommerce/bronze/raw_files/incremental
Checkpoint Path: /Volumes/ecommerce/bronze/raw_files/checkpoints/incremental_orders


**Preview incremental files (batch read)**

In [0]:
preview_df = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .csv("/Volumes/ecommerce/bronze/raw_files/incremental/*/*.csv"))

preview_df = preview_df.withColumn("source_file", preview_df["_metadata.file_path"])
display(preview_df)


order_id,order_date,customer_name,product,category,quantity,price,source_file
2002,2026-09-06,John,Mouse,Electronics,2,25,dbfs:/Volumes/ecommerce/bronze/raw_files/incremental/day2/orders_day2.csv
2003,2026-09-06,Priya,Shoes,Fashion,1,60,dbfs:/Volumes/ecommerce/bronze/raw_files/incremental/day2/orders_day2.csv
2011,2026-09-06,Ravi,Keyboard,Electronics,1,45,dbfs:/Volumes/ecommerce/bronze/raw_files/incremental/day2/orders_day2.csv
2012,2026-09-06,John,Laptop,Electronics,1,800,dbfs:/Volumes/ecommerce/bronze/raw_files/incremental/day2/orders_day2.csv
2013,2026-09-06,Priya,T-Shirt,Fashion,2,20,dbfs:/Volumes/ecommerce/bronze/raw_files/incremental/day2/orders_day2.csv
2014,2026-09-06,David,Shoes,Fashion,1,60,dbfs:/Volumes/ecommerce/bronze/raw_files/incremental/day2/orders_day2.csv
2015,2026-09-06,Anitha,Blender,Home,1,70,dbfs:/Volumes/ecommerce/bronze/raw_files/incremental/day2/orders_day2.csv
2016,2026-09-06,Robert,Mouse,Electronics,3,25,dbfs:/Volumes/ecommerce/bronze/raw_files/incremental/day2/orders_day2.csv
2017,2026-09-06,Suresh,Headphones,Electronics,2,75,dbfs:/Volumes/ecommerce/bronze/raw_files/incremental/day2/orders_day2.csv
2018,2026-09-06,Emily,Jeans,Fashion,1,40,dbfs:/Volumes/ecommerce/bronze/raw_files/incremental/day2/orders_day2.csv


**Start Auto Loader Stream (Bronze Incremental Table)**

In [0]:
from pyspark.sql import functions as F

incremental_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option(
        "cloudFiles.schemaLocation",
        "/Volumes/ecommerce/bronze/raw_files/schema/incremental_orders"
    )
    .load("/Volumes/ecommerce/bronze/raw_files/incremental/*/")
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("bronze_ingested_at", F.current_timestamp())
)

In [0]:
incremental_stream_df.writeStream \
    .format("delta") \
    .option("cloudFiles.schemaLocation", "/Volumes/ecommerce/bronze/raw_files/schema/incremental_orders") \
    .option("checkpointLocation", "/Volumes/ecommerce/bronze/raw_files/checkpoints/incremental_orders") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option("mergeSchema", "true") \
    .table("ecommerce.bronze.incremental_orders")

    


**Bronze Ingestion Validation**

In [0]:
# Load Bronze incremental table
bronze_df = spark.table("ecommerce.bronze.incremental_orders")

print("Total Records in Bronze Incremental Table:", bronze_df.count())
display(bronze_df.orderBy("order_date"))


Total Records in Bronze Incremental Table: 32


order_id,order_date,customer_name,product,category,quantity,price,_rescued_data,source_file,bronze_ingested_at,discount_percentage
2010,2026-09-05,Sarah,Backpack,Fashion,1,35,null,orders_day1.csv,2026-10-09T15:48:05.525Z,null
2003,2026-09-05,Priya,Shoes,Fashion,1,60,null,orders_day1.csv,2026-10-09T15:48:05.525Z,null
2005,2026-09-05,Anitha,T-Shirt,Fashion,2,20,null,orders_day1.csv,2026-10-09T15:48:05.525Z,null
2009,2026-09-05,Arun,Monitor,Electronics,1,250,null,orders_day1.csv,2026-10-09T15:48:05.525Z,null
2008,2026-09-05,Emily,Coffee Maker,Home,1,90,null,orders_day1.csv,2026-10-09T15:48:05.525Z,null
2001,2026-09-05,Ravi,Laptop,Electronics,1,800,null,orders_day1.csv,2026-10-09T15:48:05.525Z,null
2002,2026-09-05,John,Mouse,Electronics,1,25,null,orders_day1.csv,2026-10-09T15:48:05.525Z,null
2004,2026-09-05,David,Keyboard,Electronics,1,45,null,orders_day1.csv,2026-10-09T15:48:05.525Z,null
2006,2026-09-05,Robert,Headphones,Electronics,1,75,null,orders_day1.csv,2026-10-09T15:48:05.525Z,null
2007,2026-09-05,Suresh,Jeans,Fashion,1,40,null,orders_day1.csv,2026-10-09T15:48:05.525Z,null


**Delta History (Bronze Table)**

In [0]:
spark.sql("""
DESCRIBE HISTORY ecommerce.bronze.incremental_orders
""").show(truncate=False)


+-------+-------------------+--------------+-----------------------------+----------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+-----------------+------------------------------------+------------------------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName                     |operation   

In [0]:
#  %sql
# DROP TABLE IF EXISTS ecommerce.bronze.incremental_orders;

# DROP TABLE IF EXISTS ecommerce.silver.incremental_orders_silver;

# DROP TABLE IF EXISTS ecommerce.gold.daily_revenue;
# DROP TABLE IF EXISTS ecommerce.gold.category_revenue;
# DROP TABLE IF EXISTS ecommerce.gold.top_customers;

In [0]:
# dbutils.fs.rm(
#     "/Volumes/ecommerce/bronze/raw_files/checkpoints/incremental_orders",
#     True
# )

In [0]:
# dbutils.fs.rm(
#     "/Volumes/ecommerce/bronze/raw_files/schema/incremental_orders",
#     True
# )